In [1]:
# Libraries
import pandas as pd
import geopandas as gpd
import numpy as np 
import rasterio
from rasterio.warp import reproject, Resampling, calculate_default_transform, transform_bounds # Reprojection
from rasterio.transform import Affine

In [2]:
# Verify no issues after saving from ArcGIS Pro and check unique values
hsg_raw = './data/SSURGO_raw/CONUS_HSG_raw.tif'

unique_values = set() # Create a set to store unique values

with rasterio.open(hsg_raw, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
    
    # Check unique values
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window)
        unique_values.update(np.unique(data)) # Updates unique values to the set

# Print out unique values
print(f'Unique values: {unique_values}')

# 0 == ArcGIS Pro nodata
# 1 == A
# 2 == B
# 3 == C
# 4 == D
# 5 == A/D
# 6 == B/D
# 7 == C/D
# 15 == raster nodata

Profile: {'driver': 'GTiff', 'dtype': 'int8', 'nodata': 15.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101004,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: 15.0
CRS: EPSG:5070
Resolution (30.0, 30.0)
Unique va

In [3]:
# View raw hsg cover data - check %nodata

hsg_raw = './data/SSURGO_raw/CONUS_HSG_raw.tif'

with rasterio.open(hsg_raw, mode = 'r') as src:

    # Set total_nodata
    total_nodata = 0
    
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)

        # Count nodata cells
        total_nodata += np.sum(data.mask)
    
    # % nodata 
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

% nodata cells: 0.41285090666885854


In [6]:
# Update profile - DONE
    # int8 (int8 used as intermediary)
    # nodata = -10 (& replace nodata cells with new nodata value)

hsg_raw = './data/SSURGO_raw/CONUS_HSG_raw.tif'
hsg_profile_update = './data/SSURGO_raw/hsg/hsg_profile_update.tif'
            
with rasterio.open(hsg_raw) as src:
    profile = src.profile.copy()
    profile.update(dtype = rasterio.int8,
                   nodata = -10)

    with rasterio.open(hsg_profile_update, 'w', **profile) as dst:
        for ji, window in src.block_windows(1):
            data = src.read(1, window = window).astype(rasterio.int8) 
            
            # Replace 15 or 0 cells (previous nodata values) to -10 (new nodata value)
            data[(data == 15)|(data == 0)] = -10
            
            # Write out new raster
            dst.write(data.astype(rasterio.int8), 1, window = window)

In [8]:
# Verify profile update & unique values 

hsg_profile_update = './data/SSURGO_raw/hsg/hsg_profile_update.tif'

unique_values = set() # Create a set to store unique values

with rasterio.open(hsg_profile_update, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
    
    # Check unique values
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window)
        unique_values.update(np.unique(data)) # Updates unique values to the set

# Print out unique values
print(f'Unique values: {unique_values}')

Profile: {'driver': 'GTiff', 'dtype': 'int8', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolutio

In [9]:
# View check %nodata after profile update

hsg_profile_update = './data/SSURGO_raw/hsg/hsg_profile_update.tif'

with rasterio.open(hsg_profile_update, mode = 'r') as src:

    # Set total_nodata
    total_nodata = 0
    
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)

        # Count nodata cells
        total_nodata += np.sum(data.mask)
    
    # % nodata 
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

% nodata cells: 0.536924738320865


In [2]:
# Unzip HYSOGs250m - only run initially!!

#from zipfile import ZipFile

#HYSOGs250m_zip = './data/HYSOGs250m/Global_Hydrologic_Soil_Group_1566.zip'
#HYSOGs250m_out = r'./data/HYSOGs250m'

#with ZipFile(HYSOGs250m_zip, 'r') as zObject:
#    # Extract downloaded percent impervious data and store in data > percent_impervious folder
#    zObject.extractall(path = HYSOGs250m_out)

In [9]:
# Read in raw HYSOGs250m

HYSOGs250m = './data/HYSOGs250m/Global_Hydrologic_Soil_Group_1566/data/HYSOGs250m.tif'

with rasterio.open(HYSOGs250m, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')

# 1 == A
# 2 == B
# 3 == C
# 4 == D
# 11 == A/D
# 12 == B/D
# 13 == C/D
# 14 == D/D
# 255 = raster nodata

Profile: {'driver': 'GTiff', 'dtype': 'uint8', 'nodata': 255.0, 'width': 172800, 'height': 67200, 'count': 1, 'crs': CRS.from_wkt('GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]'), 'transform': Affine(0.002083333, 0.0, -180.0,
       0.0, -0.002083333, 83.999167206), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: 255.0
CRS: EPSG:4326
Resolution (0.002083333, 0.002083333)


In [10]:
conus = gpd.read_file('./data/CONUS shp/CONUS.shp')

# Og crs
print(f'og conus crs: {conus.crs}')

# Reproject to epsg:4326
conus = conus.to_crs('epsg:4326')

# New crs
print(f'new conus crs: {conus.crs}')

conus

og conus crs: EPSG:4269
new conus crs: epsg:4326


,Name,geometry
0,CONUS,"MULTIPOLYGON (((-118.55845 33.00683, -118.5527..."


In [13]:
# Clip HYSOGs250m to conus extent - DONE
# https://gis.stackexchange.com/questions/444062/clipping-raster-geotiff-with-a-vector-shapefile-in-python

from rasterio.mask import mask

HYSOGs250m_IN = './data/HYSOGs250m/Global_Hydrologic_Soil_Group_1566/data/HYSOGs250m.tif'
HYSOGs250m_OUT = './data/HYSOGs250m/HYSOGs250m outputs/HYSOGs250m_CONUS.tif'

with rasterio.open(HYSOGs250m_IN) as src:
    print(f'conus shp crs = HYSOGs250m crs: {conus.crs == src.crs}')
    out_image, out_transform=mask(src, conus.geometry, crop=True)
    out_meta = src.meta.copy() # Copy src metadata
    
out_meta.update({
    'driver':'Gtiff',
    'height':out_image.shape[1], # Height starts with shape[1]
    'width':out_image.shape[2], # Width starts with shape[2]
    'transform':out_transform
})
              
with rasterio.open(HYSOGs250m_OUT,'w',**out_meta) as dst:
    dst.write(out_image)

conus shp crs = HYSOGs250m crs: True


In [17]:
# Verify clipped raster

HYSOGs250m_conus = './data/HYSOGs250m/HYSOGs250m outputs/HYSOGs250m_CONUS.tif'

with rasterio.open(HYSOGs250m_conus, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')

Profile: {'driver': 'GTiff', 'dtype': 'uint8', 'nodata': 255.0, 'width': 27752, 'height': 11935, 'count': 1, 'crs': CRS.from_wkt('GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]'), 'transform': Affine(0.002083333, 0.0, -124.764592171,
       0.0, -0.002083333, 49.38458941099999), 'blockxsize': 27752, 'blockysize': 1, 'tiled': False, 'interleave': 'band'}
Nodata: 255.0
CRS: EPSG:4326
Resolution (0.002083333, 0.002083333)


In [18]:
# Update profile - DONE
    # dtype = int8 (int16 used as intermediary)
    # nodata = -10 (& replace nodata cells with new nodata value)

HYSOGs250m_conus = './data/HYSOGs250m/HYSOGs250m outputs/HYSOGs250m_CONUS.tif'
HYSOGs250m_conus_profile_update = './data/HYSOGs250m/HYSOGs250m outputs/HYSOGs250m_CONUS_profile_update.tif'

with rasterio.open(HYSOGs250m_conus) as src:
    profile = src.profile.copy()
    profile.update(dtype = rasterio.int8,
                   nodata = -10)

    with rasterio.open(HYSOGs250m_conus_profile_update, 'w', **profile) as dst:
        for ji, window in src.block_windows(1):
            data = src.read(1, window = window).astype(rasterio.int16) # int16 as intermediary to accomodate 255 in the raw HYSOGs250m
            
            # Replace 255 cells (previous nodata value) to -10 (new nodata value)
            data[data == 255] = -10
            
            # Write out new raster
            dst.write(data.astype(rasterio.int8), 1, window = window)

In [20]:
# Verify updated profile 

HYSOGs250m_conus_profile_update = './data/HYSOGs250m/HYSOGs250m outputs/HYSOGs250m_CONUS_profile_update.tif'

unique_values = set() # Create a set to store unique values

with rasterio.open(HYSOGs250m_conus_profile_update, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
    
    # Check unique values
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window)
        
        # Updates unique values to the set
        unique_values.update(np.unique(data)) 

# Print out unique values
print(f'Unique values: {sorted(unique_values)}')

Profile: {'driver': 'GTiff', 'dtype': 'int8', 'nodata': -10.0, 'width': 27752, 'height': 11935, 'count': 1, 'crs': CRS.from_wkt('GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]'), 'transform': Affine(0.002083333, 0.0, -124.764592171,
       0.0, -0.002083333, 49.38458941099999), 'blockxsize': 27752, 'blockysize': 1, 'tiled': False, 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:4326
Resolution (0.002083333, 0.002083333)
Unique values: [-10, 1, 2, 3, 4, 11, 12, 13, 14]


In [2]:
# Reclassify HYSOGs250m cell values to match SSURGO values - DONE
    # 1 = A >> 1
    # 2 = B >> 2
    # 3 = C >> 3
    # 4 = D >> 4
    # 11 = A/D >> 5
    # 12 = B/D >> 6
    # 13 = C/D >> 7
    # 14 = D/D >> 4
    
HYSOGs250m_conus_profile_update = './data/HYSOGs250m/HYSOGs250m outputs/HYSOGs250m_CONUS_profile_update.tif'
HYSOGS250m_hsg_reclassify = './data/HYSOGs250m/HYSOGs250m outputs/HYSOGs250m_CONUS_reclasify_ssurgo_match.tif'
    
with rasterio.open(HYSOGs250m_conus_profile_update, mode = 'r') as src:
    profile = src.profile.copy()
    
    with rasterio.open(HYSOGS250m_hsg_reclassify, 'w', **profile) as dst:
        for ji, window in src.block_windows(1):
            data = src.read(1, window = window)
            
            # Convert 11 (A/D) to 5 (A/D)
            data[data == 11] = 5
            
            # Convert 12 (B/D) to 6 (B/D)
            data[data == 12] = 6
            
            # Convert 13 (C/D) to 7 (C/D)
            data[data == 13] = 7
            
            # Convert 14 (D/D) to 4 (D)
            data[data == 14] = 4
            
            # Write out new raster
            dst.write(data, 1, window = window)

In [3]:
# Verify reclassification

HYSOGS250m_hsg_reclassify = './data/HYSOGs250m/HYSOGs250m outputs/HYSOGs250m_CONUS_reclasify_ssurgo_match.tif'

unique_values = set()

with rasterio.open(HYSOGS250m_hsg_reclassify, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
    
    # Check unique values
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window)
        
        # Updates unique values to the set
        unique_values.update(np.unique(data)) 

# Print out unique values
print(f'Unique values: {sorted(unique_values)}')

Profile: {'driver': 'GTiff', 'dtype': 'int8', 'nodata': -10.0, 'width': 27752, 'height': 11935, 'count': 1, 'crs': CRS.from_wkt('GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]'), 'transform': Affine(0.002083333, 0.0, -124.764592171,
       0.0, -0.002083333, 49.38458941099999), 'blockxsize': 27752, 'blockysize': 1, 'tiled': False, 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:4326
Resolution (0.002083333, 0.002083333)
Unique values: [-10, 1, 2, 3, 4, 5, 6, 7]


In [8]:
# Reproject to epsg:5070; 30m cells 

HYSOGS250m_hsg_reclassify = './data/HYSOGs250m/HYSOGs250m outputs/HYSOGs250m_CONUS_reclasify_ssurgo_match.tif'
HYSOGs250m_conus_reprojected = './data/HYSOGs250m/HYSOGs250m outputs/HYSOGs250m_CONUS_reprojected.tif' 

dst_crs = 'epsg:5070' # Target crs
res = 30 # Res in m

with rasterio.open(HYSOGS250m_hsg_reclassify) as src:
    # Calculate transform matrix for output
    dst_transform, dst_width, dst_height = calculate_default_transform(
        src.crs, 
        dst_crs, 
        src.width, 
        src.height, 
        *src.bounds, # Unpacks bounds (left, bottom, right, top)
        resolution = res
    )

    # Set output properties
    dst_profile = src.profile.copy()
    dst_profile.update(
        crs = dst_crs,
        transform = dst_transform,
        width = dst_width,
        height = dst_height,
        nodata = -10,
        tiled = True,
        blockxsize = 128,
        blockysize = 128
    )

    # Reproject each band
    with rasterio.open(HYSOGs250m_conus_reprojected, 'w', **dst_profile) as dst:
        for i in range(1, src.count + 1):
            reproject(
                source = rasterio.band(src, i),
                destination = rasterio.band(dst, i),
                src_transform = src.transform,
                src_crs = src.crs,
                dst_transform = dst_transform,
                dst_crs = dst_crs,
                resampling = Resampling.nearest # Nearest resampling for categorical data
            )

In [9]:
# Verify reprojection

HYSOGs250m_conus_reprojected = './data/HYSOGs250m/HYSOGs250m outputs/HYSOGs250m_CONUS_reprojected.tif' 

unique_values = set()

with rasterio.open(HYSOGs250m_conus_reprojected, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
    
    # Check unique values
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window)
        
        # Updates unique values to the set
        unique_values.update(np.unique(data)) 

# Print out unique values
print(f'Unique values: {sorted(unique_values)}')

Profile: {'driver': 'GTiff', 'dtype': 'int8', 'nodata': -10.0, 'width': 194954, 'height': 102975, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2910234.3735031593,
       0.0, -30.0, 3254913.3328736727), 'blockxsize': 194954, 'blockysize': 1, 'tiled': False, 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolut

In [10]:
# Verify reprojection - check %nodata

HYSOGs250m_conus_reprojected = './data/HYSOGs250m/HYSOGs250m outputs/HYSOGs250m_CONUS_reprojected.tif'

with rasterio.open(HYSOGs250m_conus_reprojected, mode = 'r') as src:

    # Set total_nodata
    total_nodata = 0
    
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)

        # Count nodata cells
        total_nodata += np.sum(data.mask)
    
    # % nodata 
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

% nodata cells: 0.5749708508624776


In [11]:
# Co-register SSURGO hsg raster with HYSOGS250m raster - DONE

# Define co-register function - NEAREST RESAMPLING
def coregister_rasters(infile, match, outfile):
    """Reproject a file to match the shape and projection of existing raster. 
    
    Parameters
    ----------
    infile : (string) path to input file to reproject
    match : (string) path to raster with desired shape and projection 
    outfile : (string) path to output file tif
    """
    # Open input
    with rasterio.open(infile) as src:
        src_transform = src.transform
        
        # Open input to match
        with rasterio.open(match) as match:
            dst_crs = match.crs
            dst_transform = match.transform # Ensures resolutions of outfile and match will be exactly the same
            dst_width = match.width
            dst_height = match.height

        # Set properties for output
        dst_kwargs = src.meta.copy()
        dst_kwargs.update({'crs': dst_crs,
                           'transform': dst_transform,
                           'width': dst_width,
                           'height': dst_height,
                           'nodata': -10})
        print('Coregistered to shape:', dst_height, dst_width,'\n Affine', dst_transform)
        
        # Open output
        with rasterio.open(outfile, "w", **dst_kwargs) as dst:
            # Iterate through bands and write using reproject function
            for i in range(1, src.count + 1):
                reproject(
                    source = rasterio.band(src, i),
                    destination = rasterio.band(dst, i),
                    src_transform = src.transform,
                    src_crs = src.crs,
                    dst_transform = dst_transform,
                    dst_crs = dst_crs,
                    resampling = Resampling.nearest)
                

# Apply coregister_rasters
HYSOGs250m_conus_reprojected = './data/HYSOGs250m/HYSOGs250m outputs/HYSOGs250m_CONUS_reprojected.tif' # Input
ref_raster = './data/SSURGO_raw/hsg/hsg_profile_update.tif' # Match
HYSOGs250m_conus_coregistered = './data/HYSOGs250m/HYSOGs250m outputs/HYSOGs250m_CONUS_voregistered_w_SSURGO_hsg.tif' # Output

coregister_rasters(
    infile = HYSOGs250m_conus_reprojected,
    match = ref_raster,
    outfile = HYSOGs250m_conus_coregistered
)

Coregistered to shape: 96751 153996 
 Affine | 30.00, 0.00,-2356125.00|
| 0.00,-30.00, 3172575.00|
| 0.00, 0.00, 1.00|


In [12]:
# Verify co-registering

HYSOGs250m_conus_coregistered = './data/HYSOGs250m/HYSOGs250m outputs/HYSOGs250m_CONUS_voregistered_w_SSURGO_hsg.tif'

unique_values = set()

with rasterio.open(HYSOGs250m_conus_coregistered, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
    
    # Check unique values
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window)
        
        # Updates unique values to the set
        unique_values.update(np.unique(data)) 

# Print out unique values
print(f'Unique values: {sorted(unique_values)}')

Profile: {'driver': 'GTiff', 'dtype': 'int8', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 153996, 'blockysize': 1, 'tiled': False, 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolution (30.0, 30.0)
Un

In [13]:
# Verify co-registering - check %nodata

HYSOGs250m_conus_coregistered = './data/HYSOGs250m/HYSOGs250m outputs/HYSOGs250m_CONUS_voregistered_w_SSURGO_hsg.tif'

with rasterio.open(HYSOGs250m_conus_coregistered, mode = 'r') as src:

    # Set total_nodata
    total_nodata = 0
    
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)

        # Count nodata cells
        total_nodata += np.sum(data.mask)
    
    # % nodata 
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

% nodata cells: 0.427312421054623


In [5]:
# Fill in NA cells in SSURGO hsg raster with HYSOGs250m values

hsg_profile_update = './data/SSURGO_raw/hsg/hsg_profile_update.tif'
HYSOGs250m_conus_coregistered = './data/HYSOGs250m/HYSOGs250m outputs/HYSOGs250m_CONUS_voregistered_w_SSURGO_hsg.tif'
hsg_final_composite = './data/SSURGO_raw/hsg/hsg_FINAL_composite.tif'

# Open SSURGO raster
with rasterio.open(hsg_profile_update) as src:
    
    # Copy profile
    profile = src.profile.copy()
    
    # Obtain nodata value
    nodata_value = profile['nodata']
    
    # Open HYSOGs250m raster
    with rasterio.open(HYSOGs250m_conus_coregistered) as src2:
        
        with rasterio.open(hsg_final_composite, 'w', **profile) as dst:
            
            for ji, window in src.block_windows(1):
                
                # SSURGO data
                ssurgo_data = src.read(1, window = window)
                
                # SSURGO mask (all nodata cells)
                ssurgo_nodata = (ssurgo_data == nodata_value)
                
                # HYSOGs250m data
                hysogs250m_data = src2.read(1, window = window)
                
                # Replace nodata cells in the SSURGO raster with the values in the HYSOGs250m raster
                ssurgo_data[ssurgo_nodata] = hysogs250m_data[ssurgo_nodata]
                
                # Write out
                dst.write(ssurgo_data, 1, window = window)

In [6]:
# Verify filling SSURGO NAs with HYSGs250m values 

hsg_final_composite = './data/SSURGO_raw/hsg/hsg_FINAL_composite.tif'

unique_values = set()

with rasterio.open(hsg_final_composite, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
    
    # Check unique values
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window)
        
        # Updates unique values to the set
        unique_values.update(np.unique(data)) 

# Print out unique values
print(f'Unique values: {sorted(unique_values)}')

Profile: {'driver': 'GTiff', 'dtype': 'int8', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolutio

In [7]:
# Verify filling SSURGO NAs with HYSOGs250m values - check % nodata

hsg_final_composite = './data/SSURGO_raw/hsg/hsg_FINAL_composite.tif'

with rasterio.open(hsg_final_composite, mode = 'r') as src:

    # Set total_nodata
    total_nodata = 0
    
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)

        # Count nodata cells
        total_nodata += np.sum(data.mask)
    
    # % nodata 
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

% nodata cells: 0.42464799883770066


In [2]:
# Reclassify hsg values to standard suitability scores
    # A (1), B (2) - standardized suitability score: 10
    # A/D (5), B/D (6) - standardized suitability score: 9
    # C (3) - standardized suitability score: 8
    # C/D (7) - standardized suitability score: 7
    # D (4) - standardized suitability score: 0
    
hsg_final_composite = './data/SSURGO_raw/hsg/hsg_FINAL_composite.tif'
hsg_final_composite_standardized = './data/SSURGO_raw/hsg/hsg_FINAL_composite_standardized.tif'
    
with rasterio.open(hsg_final_composite, mode = 'r') as src:
    profile = src.profile.copy()
    profile.update(dtype = rasterio.float32)
    
    with rasterio.open(hsg_final_composite_standardized, 'w', **profile) as dst:
        for ji, window in src.block_windows(1):
            data = src.read(1, window = window).astype(rasterio.float32)
            
            # Convert A (1), B (2) to 10
            data[(data == 1) | (data == 2)] = 10
            
            # Convert A/D (5), B/D (6) to 9
            data[(data == 5) | (data == 6)] = 9
            
            # Convert C (3) to 8
            data[data == 3] = 8
            
            # Convert C/D (7) to 7
            data[data == 7] = 7
            
            # Convert D (4) to 0
            data[data == 4] = 0
            
            # Write out new raster
            dst.write(data, 1, window = window)

In [3]:
# Verify standardized values

hsg_final_composite_standardized = './data/SSURGO_raw/hsg/hsg_FINAL_composite_standardized.tif'

with rasterio.open(hsg_final_composite_standardized, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolu

In [ ]:
# Verify standardized unique values 

hsg_final_composite_standardized = './data/SSURGO_raw/hsg/hsg_FINAL_composite_standardized.tif'

unique_values = set() # Create a set to store unique values

with rasterio.open(hsg_final_composite_standardized, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
    
    # Check unique values
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window)
        
        # Updates unique values to the set
        unique_values.update(np.unique(data)) 

# Print out unique values
print(f'Unique values: {sorted(unique_values)}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolu